# Index and RAG pipeline test

Load the saved index, inspect retrieval, and test Gemini answer generation without rebuilding the corpus.

In [1]:
from pathlib import Path
import sys

import numpy as np

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.vector_store import load_index

INDEX_PATH = ROOT / "data/vectorstore"
index = load_index(INDEX_PATH)

print("Model:", index.model_name)
print("Chunks:", len(index.chunks))
print("Embedding shape:", index.embeddings.shape)

assert len(index.chunks) == len(index.embeddings)
assert np.allclose(np.linalg.norm(index.embeddings, axis=1), 1, atol=1e-4)

Model: NeuML/pubmedbert-base-embeddings
Chunks: 466
Embedding shape: (466, 768)


Check Embedding Normalization

In [2]:
norms = np.linalg.norm(index.embeddings, axis=1)

print("Minimum norm:", norms.min())
print("Maximum norm:", norms.max())
print("Mean norm:", norms.mean())

assert np.allclose(norms, 1.0, atol=1e-4)

Minimum norm: 0.9999999
Maximum norm: 1.0000001
Mean norm: 1.0


Check LLM API Key

In [3]:
from src import config
import os

print("Gemini key configured:", bool(os.getenv("GEMINI_API_KEY")))

Gemini key configured: True


## Inspect retrieval

In [4]:
from src.vector_store import search

# question = (
#     "What was the global SARS-CoV-2 test positivity rate "
#     "for the week ending 21 June 2026?"
# )

question = "How many new leprosy cases were reported globally in 2025?"
results = search(index, question, top_k=5)

for rank, (chunk, score) in enumerate(results, start=1):
    print("\n" + "=" * 80)
    print(f"Rank {rank} | Score: {score:.4f}")
    print("Title:", chunk.metadata.get("title"))
    print("Page:", chunk.metadata.get("page_number"))
    print("Source:", chunk.metadata.get("source"))
    print(chunk.page_content[:700])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Rank 1 | Score: 0.6832
Title: Weekly Epidemiological Record, 2026, vol. 101, 36-37 [full issue]
Page: 7
Source: https://iris.who.int/handle/10665/387595
During the year, 62 countries reported zero new cases (in addition, Switzerland reported “no data available”), 53 reported 1-10
new cases, 42 reported 11-100 new cases, 30 reported 101-1000 new cases, and 12 countries reported 1001-10 000 new cases. Brazil, India and Indonesia reported more than 10 000 new cases each, together accounting for 78.3% of all new cases reported. While Brazil and Indonesia reported increase in new cases in 2025 as compared to the previous year (3.5% and 10.8% respectively), a decrease in new case detection was reported by India (9.1%). The occurrence of new leprosy cases among children (aged <15 years) is an indicator of recent transmission. A total of 8344 new
ch

Rank 2 | Score: 0.6802
Title: Weekly Epidemiological Record, 2026, vol. 101, 36-37 [full issue]
Page: 4
Source: https://iris.who.int/handle/1066

## Generate a cited answer

This cell requires `GEMINI_API_KEY` in the project `.env` file.

In [5]:
from src.rag_pipeline import answer_question

result = answer_question(
    question=question,
    index_path=INDEX_PATH,
    top_k=5,
)

print(result["answer"])

for source in result["sources"]:
    print(
        f"[{source['id']}] {source.get('title')} "
        f"page={source.get('page_number')} score={source['score']:.4f}"
    )
    print(source.get("source"))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In 2025, a total of 167,356 new leprosy cases were reported globally, with a new case detection rate of 20.3 per million population [5]. This number was 3.1% lower than the 172,717 cases reported in 2024 [5].

The 23 global priority countries accounted for 95.6% (160,042) of all new cases detected in 2025 [3]. Of the total new cases reported globally, 8,344 (5%) were among children aged less than 15 years [1]. New cases of multibacillary (MB) leprosy constituted 73.2% (122,531) of all new cases detected in 2025 [4]. Globally, 65,013 cases of leprosy were reported among women, comprising 38.8% of all new cases detected [4].
[1] Weekly Epidemiological Record, 2026, vol. 101, 36-37 [full issue] page=7 score=0.6832
https://iris.who.int/handle/10665/387595
[3] Weekly Epidemiological Record, 2026, vol. 101, 36-37 [full issue] page=6 score=0.6657
https://iris.who.int/handle/10665/387595
[4] Weekly Epidemiological Record, 2026, vol. 101, 36-37 [full issue] page=8 score=0.6515
https://iris.who.

## Check insufficient-evidence behavior

In [6]:
unsupported_result = answer_question(
    question="What was the exact rainfall total in Toronto yesterday?",
    index_path=INDEX_PATH,
    top_k=5,
)

print(unsupported_result["answer"])
print("Sources:", unsupported_result["sources"])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

The retrieved sources do not contain enough evidence to answer this question.
Sources: []
